# 3. RQ3: Reuso sem gerenciador de pacotes

**Pergunta:** como as skills se espalham, se não há registro central nem gerenciador de pacotes?

**Por que importa:** em npm ou PyPI, quando um pacote é corrigido, quem depende dele pode atualizar. Uma skill copiada é um arquivo solto: a correção na origem **não chega** às cópias. Quanto mais concentrado e espalhado o reuso, maior o efeito de uma skill problemática.

**Como medimos:** (1) concentração das cópias (coeficiente de Gini, fatia do 1% mais copiado); (2) se as cópias atravessam donos diferentes; (3) famílias de variantes (cópias editadas), base da análise de deriva do notebook 04.

---

### English summary

**Methodology step 3: quantitative analysis.**

* RQ3.1 How concentrated is reuse? Gini coefficient and share of occurrences held by the top 1% of skills.
* RQ3.2 Does copying cross owners (ecosystem reuse) or stay within one owner?
* RQ3.3 How many edited variants do popular skills have? Families are grouped by normalised skill name (baseline); `reuse.minhash_families` groups by content similarity.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("GITSKILLS_DB", str(ROOT / "data/raw/agent_skills_release.db"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 80)
from gitskills import clean, db, reuse, adoption, patterns
from gitskills.plots import save, save_table
%matplotlib inline
print("database:", os.environ["GITSKILLS_DB"])

database: /Users/cleber/Documents/gitskills-mc/data/raw/agent_skills_release.db


In [2]:
skills = clean.load_skills()
strict = clean.strict(skills)

## 3.1 Concentração do reuso
**Objetivo:** saber se o reuso é espalhado (todos copiam um pouco de tudo) ou concentrado em poucos "templates". Gini vai de 0 (todas as skills igualmente copiadas) a 1 (uma única skill concentra todas as cópias).

In [3]:
conc = pd.DataFrame([{"population": "all", **reuse.concentration(skills["copies"])},
                     {"population": "strict", **reuse.concentration(strict["copies"])}])
save_table(conc, "rq3_concentration")
conc.T

  table  -> results/tables/rq3_concentration.csv


,0,1
population,all,strict
distinct_contents,1877981,1610839
occurrences,3797117,3391374
share_singletons,0.793128,0.796137
share_occ_duplicated,0.505419,0.525019
top1pct_share_of_occ,0.258511,0.270405
top10_share_of_occ,0.002673,0.002993
gini,0.471053,0.491027
max_copies,1409,1409


**O que os dados mostram (coluna strict):**
* **79,6%** dos textos aparecem uma única vez (`share_singletons`);
* mesmo assim, **52,5%** de todas as ocorrências são cópias (`share_occ_duplicated`);
* o **1%** de textos mais copiados responde por **27%** de todas as ocorrências (`top1pct_share_of_occ`); Gini = **0,49**.

Leitura: duas populações convivem, uma cauda longa de skills únicas e um núcleo pequeno de skills copiadas em massa.

## 3.2 Reuso entre donos diferentes
**Objetivo:** distinguir duplicação interna (uma pessoa copiando entre os próprios repositórios) de reuso de ecossistema (pessoas copiando umas das outras). O dono é a conta ou organização do GitHub.

In [4]:
dup = strict[strict["copies"] > 1]
print(f"copied skills: {len(dup):,}; spread across >1 owner: {(dup.n_owners > 1).mean():.1%}")
strict.nlargest(15, "copies")[["name", "copies", "n_repos", "n_owners", "location_class", "has_scripts"]]

copied skills: 328,391; spread across >1 owner: 65.6%


,name,copies,n_repos,n_owners,location_class,has_scripts
1877946,pdf,1409,661,612,canonical,1.0
1877761,verification-before-completion,1252,914,866,canonical,0.0
1877907,frontend-design,1245,951,897,canonical,0.0
1877678,webapp-testing,1176,781,741,canonical,1.0
1877980,skill-creator,1074,29,27,skills-dir,1.0
1346245,systematic-debugging,885,656,629,canonical,1.0
1459071,canvas-design,796,550,531,canonical,0.0
1006578,receiving-code-review,795,584,560,canonical,0.0
1345091,test-driven-development,772,571,556,canonical,0.0
1287338,mcp-builder,745,555,527,canonical,1.0


**O que os dados mostram:** das 328.391 skills com mais de uma cópia, **65,6%** aparecem em mais de um dono: o reuso é majoritariamente **entre pessoas**. As mais copiadas são skills de uso geral, publicadas em coleções conhecidas (`pdf` com 1.409 cópias em 612 donos, `frontend-design`, `webapp-testing`, `test-driven-development`), e várias trazem scripts executáveis (`has_scripts = 1`). Exceção: `skill-creator` tem 1.074 cópias em só 27 donos, um caso de duplicação interna.

## 3.3 Famílias de variantes
**Objetivo:** encontrar **cópias editadas** (variantes) da mesma skill. Elas são a base do notebook 04: comparamos cada variante com a versão mais copiada da família para ver o que mudou.

**Como:** skills distintas com o mesmo `name` normalizado (minúsculas; símbolos viram hífen) formam uma família, desde que tenham pelo menos 2 textos diferentes. Detalhes técnicos em `docs/familias_de_variantes.md`.

In [5]:
fam = reuse.name_families(strict)
sizes = fam.groupby("family").agg(variants=("file_sha", "nunique"), occurrences=("copies", "sum"))
print(f"{len(sizes):,} families with >=2 variants, covering {sizes.variants.sum():,} skills")
sizes.sort_values("variants", ascending=False).head(15)

161,043 families with >=2 variants, covering 949,755 skills


,variants,occurrences
family,,
code-review,3772,6129
skill-creator,2787,10529
commit,2441,3553
review,2195,3243
frontend-design,2156,7562
release,1945,2367
brainstorming,1407,4222
systematic-debugging,1316,4097
plan,1306,2011


### Funil: da população estrita às famílias
**Objetivo:** mostrar o que acontece com **todas** as skills da população estrita, e não só com as que formam família. Só a análise de deriva (notebook 04) usa o subconjunto das famílias; concentração, reuso entre donos e prevalência de riscos usam a população estrita inteira.

In [6]:
norm = strict["name"].map(reuse.normalize_name)
named = strict[norm.notna()].assign(family=norm[norm.notna()])
n_texts = named.groupby("family")["file_sha"].transform("nunique")
solo = named[n_texts == 1]
in_fam = named[n_texts >= 2]
funnel = pd.DataFrame([
    ("strict population (notebook 01)", len(strict), int(strict["copies"].sum())),
    ("  name unusable after normalisation (non-Latin only)", int(norm.isna().sum()), int(strict.loc[norm.isna(), "copies"].sum())),
    ("  unique name: no other text shares it", len(solo), int(solo["copies"].sum())),
    ("      of which: never copied", int((solo["copies"] == 1).sum()), int(solo.loc[solo["copies"] == 1, "copies"].sum())),
    ("      of which: only identical copies", int((solo["copies"] > 1).sum()), int(solo.loc[solo["copies"] > 1, "copies"].sum())),
    ("  shared name with >= 2 texts (in families)", len(in_fam), int(in_fam["copies"].sum())),
    ("      families", in_fam["family"].nunique(), None),
    ("      variants compared in notebook 04 (minus 1 reference per family)", len(in_fam) - in_fam["family"].nunique(), None),
], columns=["group", "skills", "files (occurrences)"])
funnel["% of strict"] = (100 * funnel["skills"] / len(strict)).round(1)
funnel.loc[funnel["group"].str.contains("families$|variants compared"), "% of strict"] = None
funnel

,group,skills,files (occurrences),% of strict
0,strict population (notebook 01),1610839,3391374.0,100.0
1,name unusable after normalisation (non-Latin only),3277,5544.0,0.2
2,unique name: no other text shares it,657807,877475.0,40.8
3,of which: never copied,563108,563108.0,35.0
4,of which: only identical copies,94699,314367.0,5.9
5,shared name with >= 2 texts (in families),949755,2508355.0,59.0
6,families,161043,NaN,NaN
7,variants compared in notebook 04 (minus 1 reference per family),788712,NaN,NaN


**O que os dados mostram:** das 1.610.839 skills da população estrita:
* **949.755 (59%)** compartilham o nome com pelo menos um texto diferente e formam **161.043 famílias**. Tirando uma referência por família, sobram as **788.712 variantes** comparadas no notebook 04.
* **657.807 (41%)** têm **nome único**: nenhuma outra skill usa o mesmo nome, então não há variante com que comparar. A maioria (563.108) nunca foi copiada; **94.699** foram copiadas só de forma **idêntica**, sem nenhuma versão editada.
* **3.277 (0,2%)** têm nome só com caracteres não latinos (por exemplo `会议准备`) e ficam fora das famílias por causa da normalização. É um viés de idioma, registrado como limitação.

Essas skills não se perdem: continuam nas métricas de concentração (§3.1), reuso entre donos (§3.2) e prevalência de riscos (notebook 04, §4.1). Só não entram na análise de deriva.

**Sobre as famílias:** **161.043** famílias reúnem **949.755** skills, ou seja, **59%** da população estrita tem pelo menos uma "irmã" com o mesmo nome e texto diferente. Algumas famílias são claramente cópias editadas de skills conhecidas (`skill-creator`, `frontend-design`, `systematic-debugging`). Outras são **nomes genéricos** (`commit`, `review`, `release`, `plan`, `setup`) que juntam skills escritas do zero por pessoas diferentes.

**Atenção:** esse segundo grupo é uma limitação do método por nome. Ele trata skills independentes como variantes e tende a **inflar a deriva** medida no notebook 04. O agrupamento por similaridade de texto (`reuse.minhash_families`) corrige isso e está planejado como teste de robustez.

## Conclusão da RQ3
* O reuso é **concentrado**: 1% dos textos responde por 27% das ocorrências.
* O reuso é **de ecossistema**: 65,6% das skills copiadas atravessam donos.
* Mais da metade das skills tem variantes, e é aí que mudanças (inclusive de segurança) podem entrar sem revisão.

**Próximo passo:** o notebook 04 pergunta se essas variantes mudam o que o agente pode fazer.